In [30]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from HPC_2P_analysis.utils.config import (
    get_itr_index,
    index_to_type,
)

from HPC_2P_analysis.utils.loadData import (
    load_raw_data,
    load_screen_data,
)


def load_transition_file(
    file_path,
    *,
    screen_mode: str = "masked",
    len_position: int = 160,
) -> dict:
    """
    Load raw .mat or screened .pkl data.

    Returns
    -------
    data
        Dictionary containing:
            file_info, zones, cell_ids, firing, index.
    """
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".mat":
        return load_raw_data(
            file_path,
            len_position=len_position,
        )

    if suffix in {".pkl", ".pickle"}:
        return load_screen_data(
            file_path,
            screen_mode=screen_mode,
        )

    raise ValueError(
        f"Unsupported file type: {suffix}. "
        "Expected .mat, .pkl, or .pickle."
    )

In [31]:
def _strip_couple_prefix(track_name: str) -> str:
    """
    Convert 'couple_ACB' -> 'ACB'.
    Other names are returned unchanged.
    """
    track_name = str(track_name)

    if track_name.startswith("couple_"):
        return track_name.replace("couple_", "", 1)

    return track_name


def make_shared_track_pairs(
    couple_ana_tt: tuple[str, ...],
    pattern_ana_tt: tuple[str, ...],
) -> list[tuple[str, str]]:
    """
    Build matched Couple -> Pattern track pairs using shared base names.

    Example
    -------
    couple_ana_tt = ("couple_ACB", "couple_BCA")
    pattern_ana_tt = ("ABC", "ACB", "BAC", "BCA", "CAB", "CBA")

    Returns
    -------
    [("couple_ACB", "ACB"), ("couple_BCA", "BCA")]
    """
    pattern_set = set(pattern_ana_tt)

    pairs = []

    for couple_tt in couple_ana_tt:
        base_name = _strip_couple_prefix(couple_tt)

        if base_name in pattern_set:
            pairs.append((couple_tt, base_name))

    if len(pairs) == 0:
        raise ValueError(
            "No shared Couple-Pattern tracks found. "
            f"couple_ana_tt={couple_ana_tt}, pattern_ana_tt={pattern_ana_tt}"
        )

    return pairs


def pool_position(activity: np.ndarray, bin_size: int = 4) -> np.ndarray:
    """
    Average-pool adjacent position bins.

    Parameters
    ----------
    activity
        Shape: (n_neuron, n_position)
    bin_size
        Number of adjacent bins to average.

    Returns
    -------
    pooled
        Shape: (n_neuron, n_pooled_position)
    """
    if bin_size <= 1:
        return activity

    n_neuron, n_pos = activity.shape
    n_keep = (n_pos // bin_size) * bin_size

    if n_keep <= 0:
        raise ValueError(
            f"bin_size={bin_size} is too large for n_pos={n_pos}."
        )

    activity = activity[:, :n_keep]
    activity = activity.reshape(n_neuron, -1, bin_size).mean(axis=2)

    return activity


def extract_one_track_activity(
    data: dict,
    *,
    track_type: str,
    ana_bt: tuple[str, ...] = ("Correct",),
    position_window: tuple[int | None, int | None] | None = None,
    bin_size: int = 4,
    min_trials: int = 1,
) -> np.ndarray:
    """
    Extract trial-averaged activity for one track type.

    This function does NOT merge different tracks.

    Parameters
    ----------
    data
        Loaded data dictionary.
    track_type
        One track type, for example 'couple_ACB' or 'ACB'.
    ana_bt
        Behavior types to include.
    position_window
        Python interval [start, end). None means full track.
    bin_size
        Position pooling size.
    min_trials
        Skip condition if fewer than this many trials.

    Returns
    -------
    activity
        Shape: (n_neuron, n_position_feature)
    """
    segments = []

    for tt_idx, bt_idx in get_itr_index(data, (track_type,), ana_bt):
        fr = data["firing"][tt_idx, bt_idx]

        if fr is None or fr.shape[1] < min_trials:
            continue

        n_position = fr.shape[2]

        if position_window is None:
            start, end = 0, n_position
        else:
            start, end = position_window
            start = 0 if start is None else int(start)
            end = n_position if end is None else int(end)

        if start < 0 or end > n_position or start >= end:
            raise ValueError(
                f"Invalid position_window={position_window} "
                f"for n_position={n_position}."
            )

        seg = fr[:, :, start:end].astype(float)
        segments.append(seg)

    if len(segments) == 0:
        raise ValueError(
            f"No valid firing data found for track_type={track_type}, "
            f"ana_bt={ana_bt}."
        )

    # Merge trials only within the same track type.
    merged = np.concatenate(segments, axis=1)

    # Trial-average, keep neuron × position.
    activity = np.nanmean(merged, axis=1)

    activity = pool_position(activity, bin_size=bin_size)

    return activity


def extract_paired_track_activity(
    data: dict,
    *,
    track_pairs: list[tuple[str, str]],
    ana_bt: tuple[str, ...] = ("Correct",),
    position_window: tuple[int | None, int | None] | None = None,
    bin_size: int = 4,
    min_trials: int = 1,
) -> dict:
    """
    Extract Couple and Pattern activity using matched track pairs.

    Each pair is:
        (couple_track, pattern_track)

    Example:
        ("couple_ACB", "ACB")
        ("couple_BCA", "BCA")

    For each neuron:
        couple_activity = concat([couple_ACB_curve, couple_BCA_curve])
        pattern_activity = concat([ACB_curve, BCA_curve])

    Returns
    -------
    dict with:
        couple_activity
            Shape: (n_neuron, n_pair * n_position_feature)
        pattern_activity
            Same shape.
        track_pairs
        feature_slices
            Useful for later plotting each track segment separately.
    """
    couple_blocks = []
    pattern_blocks = []
    feature_slices = []

    cursor = 0

    for couple_tt, pattern_tt in track_pairs:
        couple_block = extract_one_track_activity(
            data,
            track_type=couple_tt,
            ana_bt=ana_bt,
            position_window=position_window,
            bin_size=bin_size,
            min_trials=min_trials,
        )

        pattern_block = extract_one_track_activity(
            data,
            track_type=pattern_tt,
            ana_bt=ana_bt,
            position_window=position_window,
            bin_size=bin_size,
            min_trials=min_trials,
        )

        if couple_block.shape != pattern_block.shape:
            raise ValueError(
                "Couple and Pattern block shape mismatch for pair "
                f"{couple_tt}->{pattern_tt}. "
                f"Couple={couple_block.shape}, Pattern={pattern_block.shape}."
            )

        width = couple_block.shape[1]

        feature_slices.append(
            {
                "couple_track": couple_tt,
                "pattern_track": pattern_tt,
                "start": cursor,
                "end": cursor + width,
                "width": width,
            }
        )

        cursor += width

        couple_blocks.append(couple_block)
        pattern_blocks.append(pattern_block)

    couple_activity = np.concatenate(couple_blocks, axis=1)
    pattern_activity = np.concatenate(pattern_blocks, axis=1)

    if couple_activity.shape != pattern_activity.shape:
        raise ValueError(
            "Final Couple and Pattern activity shape mismatch. "
            f"Couple={couple_activity.shape}, Pattern={pattern_activity.shape}."
        )

    return {
        "couple_activity": couple_activity,
        "pattern_activity": pattern_activity,
        "track_pairs": track_pairs,
        "feature_slices": feature_slices,
    }

In [32]:
def build_transition_feature(
    couple_activity: np.ndarray,
    pattern_activity: np.ndarray,
    *,
    normalize: bool = True,
    include_mean: bool = True,
    include_delta: bool = True,
    delta_weight: float = 1.0,
    min_std: float = 1e-8,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build transition features for each neuron.

    For each neuron:
        mean_activity  = (couple + pattern) / 2
        delta_activity = pattern - couple

    Final feature:
        [mean_activity, delta_weight * delta_activity]

    Parameters
    ----------
    couple_activity
        Shape: (n_neuron, n_feature)
    pattern_activity
        Shape: (n_neuron, n_feature)
    normalize
        If True, z-score each neuron using Couple + Pattern together.
        This does not separately normalize the two tasks.
    include_mean
        Include the baseline structure.
    include_delta
        Include the transition change.
    delta_weight
        Weight for delta component.
    min_std
        Exclude nearly constant neurons.

    Returns
    -------
    feature_valid
        Feature matrix after filtering invalid neurons.
    valid_mask
        Boolean mask over original neurons.
    """
    couple_activity = np.asarray(couple_activity, dtype=float)
    pattern_activity = np.asarray(pattern_activity, dtype=float)

    if couple_activity.shape != pattern_activity.shape:
        raise ValueError(
            "couple_activity and pattern_activity must have the same shape. "
            f"Got {couple_activity.shape} and {pattern_activity.shape}."
        )

    both = np.concatenate([couple_activity, pattern_activity], axis=1)
    raw_std = np.nanstd(both, axis=1)

    if normalize:
        mu = np.nanmean(both, axis=1, keepdims=True)
        sigma = np.nanstd(both, axis=1, keepdims=True)

        couple_used = (couple_activity - mu) / (sigma + 1e-12)
        pattern_used = (pattern_activity - mu) / (sigma + 1e-12)
    else:
        couple_used = couple_activity.copy()
        pattern_used = pattern_activity.copy()

    mean_activity = 0.5 * (couple_used + pattern_used)
    delta_activity = pattern_used - couple_used

    blocks = []

    if include_mean:
        blocks.append(mean_activity)

    if include_delta:
        blocks.append(delta_weight * delta_activity)

    if len(blocks) == 0:
        raise ValueError("include_mean and include_delta cannot both be False.")

    feature = np.concatenate(blocks, axis=1)

    valid_mask = np.all(np.isfinite(feature), axis=1)
    valid_mask &= np.isfinite(raw_std)
    valid_mask &= raw_std > min_std

    return feature[valid_mask], valid_mask


def standardize_feature(
    x: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Column-wise standardization.

    Pure NumPy replacement for StandardScaler.
    """
    x = np.asarray(x, dtype=float)

    mu = np.nanmean(x, axis=0, keepdims=True)
    sigma = np.nanstd(x, axis=0, keepdims=True)

    x_z = (x - mu) / (sigma + 1e-12)

    return x_z, mu, sigma


def numpy_pca(
    x: np.ndarray,
    *,
    n_pca: int = 6,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    PCA using NumPy SVD.

    Parameters
    ----------
    x
        Standardized feature matrix.
    n_pca
        Number of principal components.

    Returns
    -------
    embedding
        PCA scores.
    components
        PCA components.
    explained_variance_ratio
        Variance explained by retained PCs.
    """
    x = np.asarray(x, dtype=float)

    n_components = min(n_pca, x.shape[0] - 1, x.shape[1])

    if n_components < 1:
        raise ValueError("Not enough samples/features for PCA.")

    x_centered = x - np.mean(x, axis=0, keepdims=True)

    u, s, vt = np.linalg.svd(
        x_centered,
        full_matrices=False,
    )

    embedding = u[:, :n_components] * s[:n_components]
    components = vt[:n_components]

    eigenvalues = (s ** 2) / max(x.shape[0] - 1, 1)
    total_var = np.sum(eigenvalues)

    if total_var > 0:
        evr = eigenvalues[:n_components] / total_var
    else:
        evr = np.zeros(n_components)

    return embedding, components, evr

In [33]:
def numpy_kmeans(
    x: np.ndarray,
    *,
    n_clusters: int = 4,
    max_iter: int = 100,
    n_init: int = 10,
    random_state: int = 0,
) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Pure NumPy KMeans.
    """
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"x must be 2-D, got shape={x.shape}.")

    n_samples = x.shape[0]

    if n_samples < n_clusters:
        raise ValueError(
            f"n_samples={n_samples} is smaller than n_clusters={n_clusters}."
        )

    rng = np.random.default_rng(random_state)

    best_labels = None
    best_centers = None
    best_inertia = np.inf

    for _ in range(n_init):
        init_idx = rng.choice(n_samples, size=n_clusters, replace=False)
        centers = x[init_idx].copy()
        labels = np.full(n_samples, -1, dtype=int)

        for _ in range(max_iter):
            distance = np.sum(
                (x[:, None, :] - centers[None, :, :]) ** 2,
                axis=2,
            )

            new_labels = np.argmin(distance, axis=1)

            if np.array_equal(labels, new_labels):
                break

            labels = new_labels

            for k in range(n_clusters):
                mask = labels == k

                if np.any(mask):
                    centers[k] = np.mean(x[mask], axis=0)
                else:
                    centers[k] = x[rng.integers(0, n_samples)]

        final_distance = np.sum(
            (x[:, None, :] - centers[None, :, :]) ** 2,
            axis=2,
        )
        inertia = float(np.sum(np.min(final_distance, axis=1)))

        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = labels.copy()
            best_centers = centers.copy()

    return best_labels, best_centers, best_inertia


def _logsumexp(
    a: np.ndarray,
    axis: int = 1,
    keepdims: bool = False,
) -> np.ndarray:
    """
    Stable logsumexp.
    """
    a_max = np.max(a, axis=axis, keepdims=True)
    out = a_max + np.log(
        np.sum(np.exp(a - a_max), axis=axis, keepdims=True)
    )

    if not keepdims:
        out = np.squeeze(out, axis=axis)

    return out


def _log_gaussian_diag(
    x: np.ndarray,
    means: np.ndarray,
    variances: np.ndarray,
) -> np.ndarray:
    """
    Log probability under diagonal Gaussian components.
    """
    n_features = x.shape[1]

    diff2 = (x[:, None, :] - means[None, :, :]) ** 2

    log_det = np.sum(np.log(variances), axis=1)
    quad = np.sum(diff2 / variances[None, :, :], axis=2)

    log_prob = -0.5 * (
        n_features * np.log(2.0 * np.pi)
        + log_det[None, :]
        + quad
    )

    return log_prob


def numpy_gmm_diag(
    x: np.ndarray,
    *,
    n_components: int = 4,
    max_iter: int = 200,
    n_init: int = 5,
    reg_covar: float = 1e-5,
    tol: float = 1e-5,
    random_state: int = 0,
) -> dict:
    """
    Pure NumPy diagonal-covariance GMM.
    """
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"x must be 2-D, got shape={x.shape}.")

    n_samples, n_features = x.shape

    if n_samples < n_components:
        raise ValueError(
            f"n_samples={n_samples} is smaller than "
            f"n_components={n_components}."
        )

    best_result = None
    best_ll = -np.inf

    for init_idx in range(n_init):
        init_labels, init_means, _ = numpy_kmeans(
            x,
            n_clusters=n_components,
            max_iter=50,
            n_init=1,
            random_state=random_state + init_idx,
        )

        means = init_means.copy()
        variances = np.zeros((n_components, n_features), dtype=float)
        weights = np.zeros(n_components, dtype=float)

        global_var = np.var(x, axis=0) + reg_covar

        for k in range(n_components):
            mask = init_labels == k

            if np.any(mask):
                weights[k] = np.mean(mask)
                variances[k] = np.var(x[mask], axis=0) + reg_covar
            else:
                weights[k] = 1.0 / n_components
                variances[k] = global_var.copy()

        weights = weights / np.sum(weights)

        prev_ll = -np.inf

        for _ in range(max_iter):
            log_prob = _log_gaussian_diag(x, means, variances)
            log_prob += np.log(weights[None, :] + 1e-12)

            log_norm = _logsumexp(log_prob, axis=1, keepdims=True)
            posterior = np.exp(log_prob - log_norm)

            ll = float(np.sum(log_norm))

            if abs(ll - prev_ll) < tol:
                break

            prev_ll = ll

            nk = np.sum(posterior, axis=0) + 1e-12

            weights = nk / n_samples
            means = (posterior.T @ x) / nk[:, None]

            for k in range(n_components):
                diff = x - means[k]
                variances[k] = (
                    posterior[:, k][:, None] * diff ** 2
                ).sum(axis=0) / nk[k]

            variances = np.maximum(variances, reg_covar)

        if ll > best_ll:
            labels = np.argmax(posterior, axis=1)

            n_params = (
                (n_components - 1)
                + n_components * n_features
                + n_components * n_features
            )

            bic = -2.0 * ll + n_params * np.log(n_samples)
            aic = -2.0 * ll + 2.0 * n_params

            best_ll = ll
            best_result = {
                "labels": labels,
                "posterior": posterior,
                "weights": weights,
                "means": means,
                "variances": variances,
                "log_likelihood": ll,
                "bic": float(bic),
                "aic": float(aic),
            }

    return best_result


def cluster_transition_feature(
    feature: np.ndarray,
    *,
    method: str = "gmm",
    n_clusters: int = 4,
    k_range: range | None = None,
    n_pca: int = 6,
    random_state: int = 0,
) -> dict:
    """
    Cluster transition features.

    method:
        "kmeans"
        "gmm"
        "gmm_bic"
    """
    feature_z, feature_mu, feature_sigma = standardize_feature(feature)

    embedding, components, evr = numpy_pca(
        feature_z,
        n_pca=n_pca,
    )

    if method == "kmeans":
        labels, centers, inertia = numpy_kmeans(
            embedding,
            n_clusters=n_clusters,
            max_iter=100,
            n_init=20,
            random_state=random_state,
        )

        distance = np.sum(
            (embedding - centers[labels]) ** 2,
            axis=1,
        )

        typical_score = -distance

        return {
            "method": "kmeans",
            "labels": labels,
            "posterior": None,
            "typical_score": typical_score,
            "embedding": embedding,
            "components": components,
            "explained_variance_ratio": evr,
            "centers": centers,
            "inertia": inertia,
            "feature_z": feature_z,
        }

    if method == "gmm":
        gmm = numpy_gmm_diag(
            embedding,
            n_components=n_clusters,
            max_iter=200,
            n_init=10,
            random_state=random_state,
        )

        labels = gmm["labels"]
        posterior = gmm["posterior"]
        typical_score = posterior[np.arange(len(labels)), labels]

        return {
            "method": "gmm",
            "labels": labels,
            "posterior": posterior,
            "typical_score": typical_score,
            "embedding": embedding,
            "components": components,
            "explained_variance_ratio": evr,
            "gmm": gmm,
            "feature_z": feature_z,
        }

    if method == "gmm_bic":
        if k_range is None:
            k_range = range(2, 7)

        records = []
        best_gmm = None
        best_k = None
        best_bic = np.inf

        for k in k_range:
            if k >= embedding.shape[0]:
                continue

            gmm = numpy_gmm_diag(
                embedding,
                n_components=int(k),
                max_iter=200,
                n_init=10,
                random_state=random_state,
            )

            records.append(
                {
                    "k": int(k),
                    "bic": gmm["bic"],
                    "aic": gmm["aic"],
                    "log_likelihood": gmm["log_likelihood"],
                }
            )

            if gmm["bic"] < best_bic:
                best_bic = gmm["bic"]
                best_k = int(k)
                best_gmm = gmm

        if best_gmm is None:
            raise ValueError("No valid GMM model was fitted.")

        labels = best_gmm["labels"]
        posterior = best_gmm["posterior"]
        typical_score = posterior[np.arange(len(labels)), labels]

        return {
            "method": "gmm_bic",
            "best_k": best_k,
            "labels": labels,
            "posterior": posterior,
            "typical_score": typical_score,
            "embedding": embedding,
            "components": components,
            "explained_variance_ratio": evr,
            "gmm": best_gmm,
            "bic_table": pd.DataFrame(records).sort_values("bic"),
            "feature_z": feature_z,
        }

    raise ValueError("method must be 'kmeans', 'gmm', or 'gmm_bic'.")

In [34]:
def run_shared_track_transition_clustering(
    data: dict,
    *,
    couple_ana_tt: tuple[str, ...] = ("couple_ACB", "couple_BCA"),
    pattern_ana_tt: tuple[str, ...] = (
        "ABC", "ACB", "BAC", "BCA", "CAB", "CBA"
    ),
    track_pairs: list[tuple[str, str]] | None = None,
    ana_bt: tuple[str, ...] = ("Correct",),
    position_window: tuple[int | None, int | None] | None = None,
    bin_size: int = 4,
    min_trials: int = 1,
    normalize: bool = True,
    include_mean: bool = True,
    include_delta: bool = True,
    delta_weight: float = 1.0,
    method: str = "gmm",
    n_clusters: int = 4,
    k_range: range | None = None,
    n_pca: int = 6,
    random_state: int = 0,
    top_n: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Run Couple -> Pattern transition clustering using only shared tracks.

    Key point
    ---------
    Tracks are NOT averaged together.

    Example feature:
        Couple:
            [couple_ACB curve, couple_BCA curve]

        Pattern:
            [ACB curve, BCA curve]

    Returns
    -------
    cluster_df
        One row per valid neuron:
            array_id
            cluster
            typical_score
            PC1, PC2, ...

    typical_df
        Top representative array_id for each cluster:
            cluster
            typical_rank
            array_id
            typical_score

    result
        Intermediate objects.
    """
    if track_pairs is None:
        track_pairs = make_shared_track_pairs(
            couple_ana_tt=couple_ana_tt,
            pattern_ana_tt=pattern_ana_tt,
        )

    activity_result = extract_paired_track_activity(
        data,
        track_pairs=track_pairs,
        ana_bt=ana_bt,
        position_window=position_window,
        bin_size=bin_size,
        min_trials=min_trials,
    )

    couple_activity = activity_result["couple_activity"]
    pattern_activity = activity_result["pattern_activity"]

    n_neuron = couple_activity.shape[0]
    array_ids = np.arange(n_neuron, dtype=int)

    feature, valid_mask = build_transition_feature(
        couple_activity,
        pattern_activity,
        normalize=normalize,
        include_mean=include_mean,
        include_delta=include_delta,
        delta_weight=delta_weight,
    )

    cluster_result = cluster_transition_feature(
        feature,
        method=method,
        n_clusters=n_clusters,
        k_range=k_range,
        n_pca=n_pca,
        random_state=random_state,
    )

    valid_array_ids = array_ids[valid_mask]
    labels = cluster_result["labels"]
    typical_score = cluster_result["typical_score"]
    embedding = cluster_result["embedding"]

    cluster_df = pd.DataFrame(
        {
            "array_id": valid_array_ids,
            "cluster": labels,
            "typical_score": typical_score,
        }
    )

    for pc_idx in range(embedding.shape[1]):
        cluster_df[f"PC{pc_idx + 1}"] = embedding[:, pc_idx]

    typical_rows = []

    for cluster_id in sorted(cluster_df["cluster"].unique()):
        sub = cluster_df[cluster_df["cluster"] == cluster_id].copy()
        sub = sub.sort_values("typical_score", ascending=False)

        for rank, (_, row) in enumerate(sub.head(top_n).iterrows(), start=1):
            typical_rows.append(
                {
                    "cluster": int(cluster_id),
                    "typical_rank": int(rank),
                    "array_id": int(row["array_id"]),
                    "typical_score": float(row["typical_score"]),
                }
            )

    typical_df = pd.DataFrame(typical_rows)

    result = {
        "cluster_df": cluster_df,
        "typical_df": typical_df,
        "couple_activity": couple_activity,
        "pattern_activity": pattern_activity,
        "track_pairs": track_pairs,
        "feature_slices": activity_result["feature_slices"],
        "valid_mask": valid_mask,
        "feature": feature,
        **cluster_result,
    }

    print("=" * 72)
    print("Shared-track transition clustering finished")
    print(f"method: {method}")
    print(f"track_pairs: {track_pairs}")
    print(f"bin_size: {bin_size}")
    print(f"n_neuron total: {n_neuron}")
    print(f"n_neuron valid: {len(cluster_df)}")
    print("cluster counts:")
    print(cluster_df["cluster"].value_counts().sort_index())

    if "best_k" in cluster_result:
        print(f"best_k by BIC: {cluster_result['best_k']}")

    print(
        "explained variance ratio:",
        np.round(cluster_result["explained_variance_ratio"], 3),
    )
    print("=" * 72)

    return cluster_df, typical_df, result

In [35]:
def print_typical_ids(
    typical_df,
    *,
    top_n=10,
    id_col="array_id",
    cluster_col="cluster",
    rank_col="typical_rank",
    score_col="typical_score",
    show_score=True,
    wrap_every=10,
):
    """
    Print typical neuron array_ids by cluster without notebook truncation.

    Parameters
    ----------
    typical_df
        DataFrame containing cluster, typical_rank, array_id.
    top_n
        Print top N cells per cluster.
    id_col
        Column name for neuron array id.
    cluster_col
        Column name for cluster id.
    rank_col
        Column name for within-cluster rank.
    score_col
        Column name for score.
    show_score
        Whether to print score after each id.
    wrap_every
        Number of ids per printed line.
    """
    df = typical_df.copy()

    df[cluster_col] = df[cluster_col].astype(int)
    df[rank_col] = df[rank_col].astype(int)
    df[id_col] = df[id_col].astype(int)

    df = df[df[rank_col] <= top_n]
    df = df.sort_values([cluster_col, rank_col])

    print("=" * 80)
    print(f"Typical neuron array_ids, top {top_n} per cluster")
    print("=" * 80)

    for cluster_id in sorted(df[cluster_col].unique()):
        sub = df[df[cluster_col] == cluster_id].copy()

        print()
        print(f"Cluster {cluster_id} | n_printed = {len(sub)}")
        print("-" * 80)

        items = []

        for _, row in sub.iterrows():
            array_id = int(row[id_col])
            rank = int(row[rank_col])

            if show_score and score_col in row.index:
                score = float(row[score_col])
                items.append(f"{rank}: {array_id} ({score:.4f})")
            else:
                items.append(f"{rank}: {array_id}")

        for i in range(0, len(items), wrap_every):
            print(" | ".join(items[i:i + wrap_every]))

    print()
    print("=" * 80)

In [36]:
file_path = "../../../data/HPC_2p/screen/HP01/HP01_2024-12-13_first_pattern_screen.pkl"

data = load_transition_file(
    file_path,
    screen_mode="masked",
    len_position=160,
)

cluster_df, typical_df, transition_result = run_shared_track_transition_clustering(
    data,

    couple_ana_tt=("couple_ACB", "couple_BCA"),
    pattern_ana_tt=("ABC", "ACB", "BAC", "BCA", "CAB", "CBA"),

    # 自动只保留共有 track：
    # couple_ACB -> ACB
    # couple_BCA -> BCA
    track_pairs=None,

    ana_bt=("Correct",),

    position_window=None,
    bin_size=4,

    normalize=True,
    include_mean=True,
    include_delta=True,
    delta_weight=1.0,

    # 这里改成 BIC 自动选 K
    method="gmm_bic",
    k_range=range(2, 9),
    n_pca=10,

    random_state=0,
    top_n=10,
)

print("Best K by BIC:", transition_result["best_k"])

display(transition_result["bic_table"])
display(cluster_df.head())
display(cluster_df["cluster"].value_counts().sort_index())

print_typical_ids(
    typical_df,
    top_n=10,
    show_score=False,
    wrap_every=10,
)


Shared-track transition clustering finished
method: gmm_bic
track_pairs: [('couple_ACB', 'ACB'), ('couple_BCA', 'BCA')]
bin_size: 4
n_neuron total: 1169
n_neuron valid: 1169
cluster counts:
cluster
0     56
1     92
2     39
3    203
4    109
5    137
6    114
7    419
Name: count, dtype: int64
best_k by BIC: 8
explained variance ratio: [0.057 0.043 0.039 0.037 0.034 0.029 0.027 0.025 0.023 0.022]
Best K by BIC: 8


,k,bic,aic,log_likelihood
6,8,50435.850327,49590.178365,-24628.089183
5,7,50605.805524,49866.475546,-24787.237773
4,6,50839.270172,50206.282177,-24978.141088
3,5,51054.194563,50527.548551,-25159.774275
2,4,51255.488878,50835.184850,-25334.592425
1,3,51618.990879,51305.028833,-25590.514417
0,2,51937.357915,51729.737853,-25823.868926


,array_id,cluster,typical_score,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,0,7,0.976805,-1.992892,4.447089,3.022727,-0.257434,-2.146571,0.170152,0.011737,-1.061281,1.416097,-2.591025
1,1,7,0.800032,-0.263818,-1.876071,-1.510844,-2.516751,1.495396,-2.642131,-0.783820,-3.166221,2.691603,2.260315
2,2,7,0.975669,2.719275,-3.856898,-0.377257,-5.226143,-1.877784,-0.174305,-1.398579,1.213974,-1.221402,2.079433
3,3,1,0.906825,-2.479732,2.679937,0.408595,1.238662,-6.110253,0.649618,-0.387930,0.375266,4.979940,-0.818018
4,4,7,0.895881,-0.441770,-6.736121,-0.903130,-2.141049,1.892116,-0.555655,-3.655855,-1.375470,-2.307659,-1.711304


cluster
0     56
1     92
2     39
3    203
4    109
5    137
6    114
7    419
Name: count, dtype: int64

Typical neuron array_ids, top 10 per cluster

Cluster 0 | n_printed = 10
--------------------------------------------------------------------------------
1: 212 | 2: 207 | 3: 194 | 4: 890 | 5: 189 | 6: 808 | 7: 844 | 8: 464 | 9: 515 | 10: 391

Cluster 1 | n_printed = 10
--------------------------------------------------------------------------------
1: 96 | 2: 913 | 3: 720 | 4: 656 | 5: 525 | 6: 272 | 7: 87 | 8: 487 | 9: 1088 | 10: 872

Cluster 2 | n_printed = 10
--------------------------------------------------------------------------------
1: 512 | 2: 831 | 3: 1112 | 4: 376 | 5: 206 | 6: 1114 | 7: 250 | 8: 382 | 9: 316 | 10: 210

Cluster 3 | n_printed = 10
--------------------------------------------------------------------------------
1: 650 | 2: 314 | 3: 226 | 4: 835 | 5: 821 | 6: 326 | 7: 290 | 8: 249 | 9: 345 | 10: 916

Cluster 4 | n_printed = 10
--------------------------------------------------------------------------------
1: 378 | 2: 546 | 3: 641 | 4: 10 | 5: 97 | 6: 665 | 7